In [16]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_FILE = Path("../data/cleaned_job_placement_data.csv")
OUTPUT_FILE = Path("../data/engineered_job_placement_data.csv")

df = pd.read_csv(DATA_FILE)
print("Starting shape:", df.shape)
df.head()

Starting shape: (50000, 26)


,age_years,gender,ssc_percentage,hsc_percentage,degree_percentage,degree_specialization,technical_score,aptitude_score,communication_score,skills_match_percentage,...,expected_ctc_lpa,company_tier,job_role_match,competition_level,bond_requirement,notice_period_days,layoff_history,employment_gap_months,relocation_willingness,status
0,27,Male,65.061656,83.842578,75.856526,Computer Science,58.221909,89.566305,64.474484,79.548913,...,5.805850,Tier 3,Not Matched,Medium,Not Required,15.0,No,18.0,Not Willing,Not Placed
1,24,Male,67.885626,64.973305,73.093588,Electronics,71.927978,54.591971,61.077306,73.316134,...,10.807739,Tier 1,Matched,High,Required,0.0,Unknown,0.0,Not Willing,Not Placed
2,33,Female,73.892471,68.834121,90.196460,Information Technology,72.445041,58.587088,79.494739,75.466980,...,5.476839,Tier 3,Not Matched,Low,Not Required,0.0,No,3.0,Not Willing,Placed
3,31,Male,74.145568,76.255126,75.586731,Mechanical,78.855676,61.022065,53.740386,73.676449,...,9.980580,Tier 2,Matched,Low,Not Required,0.0,Yes,6.0,Unknown,Not Placed
4,28,Male,60.475937,65.786336,80.801010,Information Technology,68.286776,65.713731,61.438314,88.994847,...,8.921233,Tier 2,Matched,Medium,Not Required,0.0,No,3.0,Willing,Not Placed


### Group candidates by years of experience

In [17]:
def create_experience_category(data):
    engineered = data.copy()
    conditions = [
        engineered["years_of_experience"].eq(0),
        engineered["years_of_experience"].between(1, 2),
        engineered["years_of_experience"].between(3, 5)
    ]
    engineered["experience_category"] = np.select(
        conditions, ["Fresher", "Junior", "Senior"], default="Unknown"
    )
    return engineered

df = create_experience_category(df)
display(df["experience_category"].value_counts())
display(pd.crosstab(df["experience_category"], df["status"]))

experience_category
Junior     23474
Fresher    14952
Senior     11574
Name: count, dtype: int64

status,Not Placed,Placed
experience_category,,
Fresher,12891,2061
Junior,16618,6856
Senior,5361,6213


### Academic performance bands

In [18]:
def create_academic_performance_band(data):
    engineered_data = data.copy()

    conditions = [
        engineered_data["degree_percentage"] < 60,
        engineered_data["degree_percentage"].between(60, 75, inclusive="left"),
        engineered_data["degree_percentage"] >= 75
    ]

    categories = ["Low", "Medium", "High"]

    engineered_data["academic_performance_band"] = pd.Series(
        "Unknown", index=engineered_data.index
    )

    engineered_data.loc[conditions[0], "academic_performance_band"] = categories[0]
    engineered_data.loc[conditions[1], "academic_performance_band"] = categories[1]
    engineered_data.loc[conditions[2], "academic_performance_band"] = categories[2]

    return engineered_data


df = create_academic_performance_band(df)

print(df["academic_performance_band"].value_counts())

print("\nPlacement status by academic performance:")
print(pd.crosstab(
    df["academic_performance_band"],
    df["status"]
))


academic_performance_band
Medium    26846
High      22078
Low        1076
Name: count, dtype: int64

Placement status by academic performance:
status                     Not Placed  Placed
academic_performance_band                    
High                            15452    6626
Low                               734     342
Medium                          18684    8162


### Skills match level

In [19]:
def create_skills_match_level(data):
    engineered_data = data.copy()

    conditions = [
        engineered_data["skills_match_percentage"] < 60,
        engineered_data["skills_match_percentage"].between(
            60, 80, inclusive="left"
        ),
        engineered_data["skills_match_percentage"] >= 80
    ]

    categories = ["Low", "Medium", "High"]

    engineered_data["skills_match_level"] = pd.Series(
        "Unknown", index=engineered_data.index
    )

    engineered_data.loc[conditions[0], "skills_match_level"] = categories[0]
    engineered_data.loc[conditions[1], "skills_match_level"] = categories[1]
    engineered_data.loc[conditions[2], "skills_match_level"] = categories[2]

    return engineered_data


df = create_skills_match_level(df)


print(df["skills_match_level"].value_counts())

print("\nPlacement status by skills match level:")
print(pd.crosstab(
    df["skills_match_level"],
    df["status"]
))


skills_match_level
Medium    28355
High      15440
Low        6205
Name: count, dtype: int64

Placement status by skills match level:
status              Not Placed  Placed
skills_match_level                    
High                      9362    6078
Low                       5519     686
Medium                   19989    8366


### Interview performance category

In [20]:
def create_interview_performance_category(data):
    engineered_data = data.copy()

    score_columns = [
        "technical_score",
        "aptitude_score",
        "communication_score"
    ]

    engineered_data["average_interview_score"] = (
        engineered_data[score_columns].mean(axis=1)
    )

    conditions = [
        engineered_data["average_interview_score"] < 60,
        engineered_data["average_interview_score"].between(
            60, 75, inclusive="left"
        ),
        engineered_data["average_interview_score"] >= 75
    ]

    categories = ["Low", "Medium", "High"]

    engineered_data["interview_performance_category"] = pd.Series(
        "Unknown", index=engineered_data.index
    )

    engineered_data.loc[conditions[0], "interview_performance_category"] = categories[0]
    engineered_data.loc[conditions[1], "interview_performance_category"] = categories[1]
    engineered_data.loc[conditions[2], "interview_performance_category"] = categories[2]

    return engineered_data


df = create_interview_performance_category(df)


print(df["interview_performance_category"].value_counts())

print("\nPlacement status by interview performance:")
print(pd.crosstab(
    df["interview_performance_category"],
    df["status"]
))


interview_performance_category
Medium    38115
Low        8202
High       3683
Name: count, dtype: int64

Placement status by interview performance:
status                          Not Placed  Placed
interview_performance_category                    
High                                  1535    2148
Low                                   7674     528
Medium                               25661   12454


## Review and save engineered features

In [21]:
new_features = [
    "experience_category",
    "academic_performance_band",
    "skills_match_level",
    "average_interview_score",
    "interview_performance_category"
]

print("Dataset shape:", df.shape)
display(df[new_features].head())
display(df[new_features].isna().sum().to_frame("Missing Values"))
display(df["status"].value_counts())
display((df["status"].value_counts(normalize=True) * 100).round(2))

OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUTPUT_FILE, index=False)
print("Engineered dataset saved to:", OUTPUT_FILE)

Dataset shape: (50000, 31)


,experience_category,academic_performance_band,skills_match_level,average_interview_score,interview_performance_category
0,Junior,High,Medium,70.754233,Medium
1,Fresher,Medium,Medium,62.532418,Medium
2,Junior,High,Medium,70.175622,Medium
3,Fresher,High,Medium,64.539376,Medium
4,Fresher,High,High,65.146274,Medium


,Missing Values
experience_category,0
academic_performance_band,0
skills_match_level,0
average_interview_score,0
interview_performance_category,0


status
Not Placed    34870
Placed        15130
Name: count, dtype: int64

status
Not Placed    69.74
Placed        30.26
Name: proportion, dtype: float64

Engineered dataset saved to: ../data/engineered_job_placement_data.csv
